In [ ]:
!pip install bertopic spacy umap-learn plotly google-generativeai gensim hdbscan
!python -m spacy download en_core_web_sm

In [ ]:
import json
import google.generativeai as genai
import spacy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import re
from bertopic import BERTopic
from umap import UMAP
from sklearn.feature_extraction.text import CountVectorizer
from gensim.models.coherencemodel import CoherenceModel
from gensim.corpora.dictionary import Dictionary
from hdbscan import HDBSCAN

def clean_html_xml_tags(text):
    if not text:
        return ""
    cleaned_text = re.sub(r'<.*?>', ' ', text)
    cleaned_text = re.sub(r'\s+', ' ', cleaned_text).strip()
    return cleaned_text

class BERTopicService:
    def __init__(self, n_topics=None, use_approx_dist=False, use_lemmatized_input=False):
        self.n_topics = n_topics
        self.use_approx_dist = use_approx_dist
        self.use_lemmatized_input = use_lemmatized_input
        self.nlp = None
        self.topic_model = None
        self.topics = None
        self.probs = None

        try:
            self.nlp = spacy.load("en_core_web_sm", disable=['parser', 'ner'])
            self.nlp.max_length = 10000000
            self._setup_stopwords()
        except OSError:
            print("Spacy model 'en_core_web_sm' not found.") # Changed logger.error to print
            raise

        self.vectorizer_model = CountVectorizer(
            tokenizer=self.spacy_tokenizer,
            ngram_range=(1, 2),
            min_df=5,
            max_df=0.7
        )

    def _setup_stopwords(self):
        academic_stopwords = [
            'finding', 'findings', 'illustrate', 'significant', 'provide', 'provides', 'potential', 'associated', 'effective', 'aspect', 'aspects', 'challenge', 'challenges',
            'paper', 'study', 'research', 'result', 'results', 'method', 'methodology',
            'proposed', 'propose', 'approach', 'based', 'using', 'used', 'use',
            'analysis', 'model', 'system', 'data', 'application', 'new', 'development',
            'performance', 'conclusion', 'abstract', 'introduction', 'work', 'time',
            'significant', 'shown', 'show', 'demonstrate', 'experiment', 'experimental',
            'university', 'department', 'author', 'et', 'al', 'figure', 'table', 'show'
            'high', 'low', 'large', 'small', 'different', 'various', 'property', 'properties', 'increase', 'show', 'effect', 'high', 'activity',
            'structure', 'compound', 'condition', 'quality', 'entry', 'contain', 'parameter', 'observe', 'report', 'present', 'evaluate'
        ]
        for word in academic_stopwords:
            self.nlp.vocab[word].is_stop = True

    def spacy_tokenizer(self, text):
        if not text: return []
        doc = self.nlp(text)
        return [token.lemma_.lower() for token in doc
                if not token.is_stop and not token.is_punct and not token.like_num and len(token) > 2]

    def fit_transform(self, documents):
        print(f"Training BERTopic (Target Topics: {self.n_topics if self.n_topics else 'Auto'})...")

        if self.use_approx_dist:
            print("Mode: approximate_distribution (c-TF-IDF for Soft Clustering)")
        else:
            print("Mode: calculate_probabilities (HDBSCAN for Confidence Score)")

        if self.use_lemmatized_input:
            print("Input: Lemmatized Text (Applying Spacy preprocessing before embedding)...")
            train_docs = []
            for doc in documents:
                tokens = self.spacy_tokenizer(doc)
                train_docs.append(" ".join(tokens))
        else:
            print("Input: Raw Text (Default for Specter 2)")
            train_docs = documents

        umap_model = UMAP(
            n_neighbors=15,
            n_components=5,
            min_dist=0.0,
            metric='cosine',
            random_state=42,
            low_memory=True
        )

        self.topic_model = BERTopic(
            umap_model=umap_model,
            embedding_model="allenai/specter2_base",
            vectorizer_model=self.vectorizer_model,
            calculate_probabilities=not self.use_approx_dist,
            nr_topics=self.n_topics if self.n_topics else None,
            verbose=True,
            top_n_words=15
        )

        self.topics, self.probs = self.topic_model.fit_transform(train_docs)

        if self.use_approx_dist:
            print("Calculating approximate topic distributions (c-TF-IDF)...")
            topic_distr, _ = self.topic_model.approximate_distribution(
                train_docs,
                min_similarity=0.01
            )

            row_sums = topic_distr.sum(axis=1)
            row_sums[row_sums == 0] = 1
            topic_distr = topic_distr / row_sums[:, np.newaxis]

            self.probs = topic_distr

        return self.topics, self.probs

    def get_top_words_list(self, n_top_words=10):
        topics_words = []
        n_valid_topics = len(self.probs[0]) if self.probs is not None and len(self.probs) > 0 else 0

        for t_id in range(n_valid_topics):
            if t_id in self.topic_model.get_topics():
                words = [word for word, _ in self.topic_model.get_topic(t_id)[:n_top_words]]
                topics_words.append(words)
            else:
                topics_words.append([])
        return topics_words

    def calculate_topic_diversity(self, n_top_words=10):
        topics_words = self.get_top_words_list(n_top_words)
        unique_words = set()
        total_words = 0
        for topic in topics_words:
            if not topic: continue
            unique_words.update(topic)
            total_words += len(topic)
        if total_words == 0: return 0
        return len(unique_words) / total_words

with open("papers_export.json", "r", encoding="utf-8") as f:
    papers = json.load(f)

docs = []
paper_ids = []

for p in papers:
    paper_ids.append(p['id'])

    clean_abstract = clean_html_xml_tags(p['abstract'])
    full_text = f"{p['title']}. {clean_abstract}"
    docs.append(full_text)

print(f"Training BERTopic on {len(docs)} documents...")
service = BERTopicService(n_topics=None, use_approx_dist=True, use_lemmatized_input=False)
topics, probs = service.fit_transform(docs)
topic_model = service.topic_model

print("BERTopic Training Completed!")

In [ ]:
def generate_llm_names(topic_model, api_key):
        if not api_key:
            print("No Gemini API key provided. Falling back to default keywords.")
            return {}

        print("Calling Gemini LLM to generate topic names...")
        genai.configure(api_key=api_key)

        model = genai.GenerativeModel('gemini-flash-latest')

        topic_info = topic_model.get_topic_info()
        prompt = (
            "You are an expert academic researcher and librarian. "
            "I have a list of topics generated by a topic modeling algorithm from research papers. "
            "For each topic, I will provide the top 15 keywords"
            "Your task is to create a concise, professional academic category name (2-5 words) for each topic. "
            "Respond ONLY with a valid JSON format mapping the Topic ID (string) to the generated name.\n"
            "Example: {\"0\": \"Clinical Stroke Management\", \"1\": \"Quantum Computing Algorithms\"}\n\n"
        )

        for _, row in topic_info.iterrows():
            topic_id = row['Topic']
            if topic_id == -1:
                continue

            keywords = [word for word, _ in topic_model.get_topic(topic_id)][:15]
            rep_docs = topic_model.get_representative_docs(topic_id)
            titles = [doc[:150] + "..." for doc in rep_docs[:3]] if rep_docs else []

            prompt += f"Topic ID: {topic_id}\n"
            prompt += f"Keywords: {', '.join(keywords)}\n"
            prompt += f"Representative Titles: {titles}\n\n"

        try:
            response = model.generate_content(
                prompt,
                generation_config=genai.GenerationConfig(
                    response_mime_type="application/json"
                )
            )
            llm_mapping = json.loads(response.text)
            print("LLM successfully generated topic names!")
            return llm_mapping

        except Exception as e:
            print(f"Failed to parse LLM response: {e}")
            return {}

API_KEY = ""
print("Assigning LLM Names...")
llm_names = generate_llm_names(topic_model, API_KEY)

results = []
for i, paper_id in enumerate(paper_ids):
    topic_id = topics[i]
    paper_prob = probs[i].tolist() if probs is not None else []

    if topic_id == -1:
        cluster_label = "Outlier / Noise"
    else:
        topic_str_id = str(topic_id)
        cluster_label = f"Topic {topic_id}: {llm_names.get(topic_str_id, ', '.join([w for w, _ in topic_model.get_topic(topic_id)][:5]))}"

    multi_labels = [cluster_label]
    if topic_id != -1 and len(paper_prob) > 0:
        for alt_topic_id, prob in enumerate(paper_prob):
            if alt_topic_id != topic_id and prob > 0.15:
                alt_str_id = str(alt_topic_id)
                alt_label = f"Topic {alt_topic_id}: {llm_names.get(alt_str_id, ', '.join([w for w, _ in topic_model.get_topic(alt_topic_id)][:5]))}"
                multi_labels.append(alt_label)

    raw_keywords = [word for word, _ in topic_model.get_topic(topic_id)][:10] if topic_id != -1 else []

    results.append({
        "id": paper_id,
        "cluster_id": topic_id,
        "cluster_label": cluster_label,
        "predicted_multi_labels": multi_labels,
        "topic_keywords": raw_keywords,
        "topic_distribution": paper_prob
    })

with open("bertopic_results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=4)
print("Successfully saved results to bertopic_results.json!")